In [139]:
## load packages 
import pandas as pd
import re
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

## nltk imports
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random

In [186]:
#defining functions

#defining a function for basic transformations that are applied to all datasets
def transpose_reset(df, to_transpose = True):
    if to_transpose:
        df = df.transpose() #transposing data since all datasets come with census tracts in the columns
    df = df.reset_index()#resetting indices so the column names are the column indices rather than being their own row
    df.columns = df.iloc[0]
    df = df.drop(0)
    return df

#defining a function for column renaming and row filtering that are applied only to four of the ACS datasets (the rent and housing price datasets)
def transform_ACS_df(df):
    df = transpose_reset(df) #including the basic transformations function in this function so that multiple calls are not necessary
    df.rename(columns={'Label (Grouping)': 'census_tract'}, inplace=True) #renaming the census tract column with a clearer name
    df = df[df['census_tract'].str.contains("!!Estimate", regex=False)] #filtering to only include rows with the actual estimated numbers for each tract, since each tract has both an estimate row and a margin of error row, which we are filtering out
    df['tract_num'] = (
        df['census_tract']
        .str.extract(r'Census Tract (\d+\.?\d*)')[0]
        .astype(float)
        .round(2)
    )
    return df

#defining a function to calculate racial proportions
def race_percentages(df, white_col, black_col, asian_col, other_col, hispanic_col, total_col):
    df['white_pct'] = df[white_col] / df[total_col] * 100
    df['black_pct'] = df[black_col] / df[total_col] * 100
    df['asian_pct'] = df[asian_col] / df[total_col] * 100
    df['other_pct'] = df[other_col] / df[total_col] * 100
    df['hispanic_pct'] = df[hispanic_col] / df[total_col] * 100
    return df

In [141]:
#pull 2020 data
df2020 = nj_censustracts_2020 = pd.read_csv('../data/DECENNIALDP2020.DP1-2026-05-14T152641(2).csv')

In [142]:
df2020.head()

,census_tract,Census Tract 1.01; Hudson County; New Jersey!!Count,Census Tract 1.01; Hudson County; New Jersey!!Percent,Census Tract 1.02; Hudson County; New Jersey!!Count,Census Tract 1.02; Hudson County; New Jersey!!Percent,Census Tract 2; Hudson County; New Jersey!!Count,Census Tract 2; Hudson County; New Jersey!!Percent,Census Tract 3; Hudson County; New Jersey!!Count,Census Tract 3; Hudson County; New Jersey!!Percent,Census Tract 4; Hudson County; New Jersey!!Count,...,Census Tract 199; Hudson County; New Jersey!!Count,Census Tract 199; Hudson County; New Jersey!!Percent,Census Tract 200; Hudson County; New Jersey!!Count,Census Tract 200; Hudson County; New Jersey!!Percent,Census Tract 201; Hudson County; New Jersey!!Count,Census Tract 201; Hudson County; New Jersey!!Percent,Census Tract 324; Hudson County; New Jersey!!Count,Census Tract 324; Hudson County; New Jersey!!Percent,Census Tract 9801; Hudson County; New Jersey!!Count,Census Tract 9801; Hudson County; New Jersey!!Percent
0,total_pop,"2,554",100.00%,"3,834",100.00%,"5,391",100.00%,"3,948",100.00%,"3,973",...,"5,542",100.00%,"5,303",100.00%,"4,256",100.00%,"6,762",100.00%,5.0,100.00%
1,under_5,160,6.30%,211,5.50%,273,5.10%,198,5.00%,247,...,335,6.00%,190,3.60%,469,11.00%,382,5.60%,1.0,20.00%
2,5_to_9,161,6.30%,205,5.30%,345,6.40%,215,5.40%,258,...,251,4.50%,187,3.50%,197,4.60%,440,6.50%,0.0,0.00%
3,10 to 14 years,171,6.70%,215,5.60%,301,5.60%,201,5.10%,205,...,258,4.70%,271,5.10%,115,2.70%,477,7.10%,1.0,20.00%
4,15 to 19 years,146,5.70%,211,5.50%,282,5.20%,185,4.70%,196,...,253,4.60%,310,5.80%,83,2.00%,450,6.70%,0.0,0.00%


In [143]:
#Applying initial cleaning
df2020 = transpose_reset(df2020)
df2020.head()

,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,"Sold, not occupied","For seasonal, recreational, or occasional use",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units
1,Census Tract 1.01; Hudson County; New Jersey!!...,"2,554",160,161,171,146,173,265,247,213,...,6,7,27,NaN,3,4.4,NaN,847,321,526
2,Census Tract 1.01; Hudson County; New Jersey!!...,100.00%,6.30%,6.30%,6.70%,5.70%,6.80%,10.40%,9.70%,8.30%,...,0.70%,0.80%,2.90%,NaN,(X),(X),NaN,100.00%,37.90%,62.10%
3,Census Tract 1.02; Hudson County; New Jersey!!...,"3,834",211,205,215,211,271,350,426,322,...,2,3,39,NaN,1.4,5.5,NaN,"1,330",433,897
4,Census Tract 1.02; Hudson County; New Jersey!!...,100.00%,5.50%,5.30%,5.60%,5.50%,7.10%,9.10%,11.10%,8.40%,...,0.10%,0.20%,2.70%,NaN,(X),(X),NaN,100.00%,32.60%,67.40%
5,Census Tract 2; Hudson County; New Jersey!!Count,"5,391",273,345,301,282,393,519,561,437,...,5,0,54,NaN,3.3,4.5,NaN,"2,079",519,"1,560"


In [144]:
#Eliminating rows that count demographic totals. Only keeping rows that count percentages
df2020 = df2020[df2020.census_tract.str.contains("!!Count")]

In [145]:
#Creating boolean variable near_hblr and listing all census tracts that count as near_hblr
keywords = [' 73;', ' 75;', ' 74;', ' 76.01;', ' 76.02;', ' 77.01;', ' 77.02;', ' 77.03;', ' 193;', ' 194;', ' 115;', ' 114;', ' 113;', ' 112;', ' 110;', ' 109;', ' 107.01;', ' 107.02;', ' 104;', ' 103;', ' 102;', ' 63;', ' 62;', ' 60;', ' 58.01;', ' 55;', ' 53;', ' 68;', ' 44;', ' 45;', ' 46;', ' 47;', ' 48;', ' 49;', ' 42;', ' 190;', ' 191;', ' 192;', ' 3;', ' 8;', ' 189;', ' 185.01;', ' 185.02;', ' 179;', ' 146;', ' 160;', ' 162;', ' 158.02;', ' 161;']
df2020.loc[:, 'near_hblr'] = df2020['census_tract'].str.contains('|'.join(keywords), na=False)

C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\3350096388.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2020.loc[:, 'near_hblr'] = df2020['census_tract'].str.contains('|'.join(keywords), na=False)


In [146]:
df2020.head()

,census_tract,total_pop,under_5,5_to_9,10 to 14 years,15 to 19 years,20 to 24 years,25 to 29 years,30 to 34 years,35 to 39 years,...,"For seasonal, recreational, or occasional use",All other vacants,VACANCY RATES,Homeowner vacancy rate (percent) [4],Rental vacancy rate (percent) [5],HOUSING TENURE,Occupied housing units,Owner-occupied housing units,Renter-occupied housing units,near_hblr
1,Census Tract 1.01; Hudson County; New Jersey!!...,"2,554",160,161,171,146,173,265,247,213,...,7,27,NaN,3,4.4,NaN,847,321,526,False
3,Census Tract 1.02; Hudson County; New Jersey!!...,"3,834",211,205,215,211,271,350,426,322,...,3,39,NaN,1.4,5.5,NaN,"1,330",433,897,False
5,Census Tract 2; Hudson County; New Jersey!!Count,"5,391",273,345,301,282,393,519,561,437,...,0,54,NaN,3.3,4.5,NaN,"2,079",519,"1,560",False
7,Census Tract 3; Hudson County; New Jersey!!Count,"3,948",198,215,201,185,292,476,420,384,...,11,55,NaN,2.1,4.9,NaN,"1,589",497,"1,092",True
9,Census Tract 4; Hudson County; New Jersey!!Count,"3,973",247,258,205,196,280,444,376,311,...,0,58,NaN,3.6,4.8,NaN,"1,419",453,966,False


In [147]:
#checking column names
print(df2020.columns.tolist())

['census_tract', 'total_pop', 'under_5', '5_to_9', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa010 to 14 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa015 to 19 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa020 to 24 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa025 to 29 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa030 to 34 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa035 to 39 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa040 to 44 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa045 to 49 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa050 to 54 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa055 to 59 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa060 to 64 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa065 to 69 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa070 to 74 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa075 to 79 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa080 to 84 years', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa085 years and over', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Selected Age Categories', '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa016 years and over', '\xa0\xa0\xa0\xa0\

In [148]:
#cleaning column names
df2020.columns = df2020.columns.str.replace('\xa0', '', regex=False)
print(df2020.columns.tolist())

['census_tract', 'total_pop', 'under_5', '5_to_9', '10 to 14 years', '15 to 19 years', '20 to 24 years', '25 to 29 years', '30 to 34 years', '35 to 39 years', '40 to 44 years', '45 to 49 years', '50 to 54 years', '55 to 59 years', '60 to 64 years', '65 to 69 years', '70 to 74 years', '75 to 79 years', '80 to 84 years', '85 years and over', 'Selected Age Categories', '16 years and over', '18 years and over', '21 years and over', '62 years and over', '65 years and over', 'Male population', 'Under 5 years', '5 to 9 years', '10 to 14 years', '15 to 19 years', '20 to 24 years', '25 to 29 years', '30 to 34 years', '35 to 39 years', '40 to 44 years', '45 to 49 years', '50 to 54 years', '55 to 59 years', '60 to 64 years', '65 to 69 years', '70 to 74 years', '75 to 79 years', '80 to 84 years', '85 years and over', 'Selected Age Categories', '16 years and over', '18 years and over', '21 years and over', '62 years and over', '65 years and over', 'Female population', 'Under 5 years', '5 to 9 years

In [149]:
#giving repeat column names distinct labels
cols = list(df2020.columns)
seen = {}
for i, col in enumerate(cols):
    if col in seen:
        seen[col] += 1
        cols[i] = f"{col}_{seen[col]}"
    else:
        seen[col] = 0
df2020.columns = cols

In [150]:
#cleaning and converting values into floats
df2020['total_pop'] = df2020['total_pop'].str.replace(',', '', regex=False).astype(float)
df2020['total_white'] = df2020['White alone_1'].str.replace(',', '', regex=False).astype(float)
df2020['total_black'] = df2020['Black or African American alone_1'].str.replace(',', '', regex=False).astype(float)
df2020['total_asian'] = df2020['Asian alone_1'].str.replace(',', '', regex=False).astype(float)
df2020['total_other'] = df2020['Some Other Race alone_1'].str.replace(',', '', regex=False).astype(float)
df2020['total_hispanic'] = df2020['Hispanic or Latino'].str.replace(',', '', regex=False).astype(float)

C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\1034329400.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2020['total_pop'] = df2020['total_pop'].str.replace(',', '', regex=False).astype(float)
C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\1034329400.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2020['total_white'] = df2020['White alone_1'].str.replace(',', '', regex=False).astype(float)
C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\1034329400.py:4: SettingWithCopyWarning: 
A value 

In [151]:
#calculating proportions using function
df2020 = race_percentages(df2020, 'total_white', 'total_black', 'total_asian', 'total_other', 'total_hispanic', 'total_pop')

C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\3746987367.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['white_pct'] = df[white_col] / df[total_col] * 100
C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\3746987367.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['black_pct'] = df[black_col] / df[total_col] * 100
C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\3746987367.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

In [152]:
#checking columns
df2020[['census_tract', 'white_pct', 'black_pct', 'asian_pct', 'other_pct', 'hispanic_pct', 'near_hblr']].head(20)

,census_tract,white_pct,black_pct,asian_pct,other_pct,hispanic_pct,near_hblr
1,Census Tract 1.01; Hudson County; New Jersey!!...,17.815192,4.111198,33.829287,1.370399,41.464370,False
3,Census Tract 1.02; Hudson County; New Jersey!!...,18.309859,5.685968,32.159624,0.808555,41.471049,False
5,Census Tract 2; Hudson County; New Jersey!!Count,15.933964,5.935819,19.347060,1.595251,54.795029,False
7,Census Tract 3; Hudson County; New Jersey!!Count,31.636272,3.571429,12.563323,1.063830,48.378926,True
9,Census Tract 4; Hudson County; New Jersey!!Count,16.763151,3.674805,41.631009,1.157815,34.533098,False
11,Census Tract 5; Hudson County; New Jersey!!Count,22.168172,4.563354,30.345147,1.343526,38.846421,False
13,Census Tract 6; Hudson County; New Jersey!!Count,21.364421,4.939551,27.668394,1.278066,42.314335,False
15,Census Tract 7; Hudson County; New Jersey!!Count,26.461624,4.682738,13.521751,1.274591,51.648656,False
17,Census Tract 8; Hudson County; New Jersey!!Count,38.822086,4.368098,13.276074,1.668712,38.331288,True
19,Census Tract 9.02; Hudson County; New Jersey!!...,19.784597,4.101505,60.106226,0.929478,13.219239,False


In [153]:
#checking percentages for each race by proximity to hblr
df2020.groupby("near_hblr")[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

,near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,27.687152,7.378781,16.944176,43.607810,1.362677
1,True,32.456182,16.887438,16.327780,30.243354,1.032812


In [163]:
#pull 2010 data
df2010 = pd.read_csv("../data/DECENNIALSF12010.P9-2026-05-19T020157 (2010).csv")

In [164]:
#initial cleaning using function
df2010 = transpose_reset(df2010)
df2010.head()

,Label (Grouping),Total:,Hispanic or Latino,White alone,Black or African American alone,American Indian and Alaska Native alone,Asian alone,Native Hawaiian and Other Pacific Islander alone,Some Other Race alone,Two or More Races:
1,"Census Tract 1, Hudson County, New Jersey","6,025","2,458","1,437",253,15,"1,693",9,32,128
2,"Census Tract 2, Hudson County, New Jersey","5,409","3,362",987,310,12,630,0,31,77
3,"Census Tract 3, Hudson County, New Jersey","4,220 (r46394)","2,346","1,267",177,10,312,0,35,73
4,"Census Tract 4, Hudson County, New Jersey","3,991","1,297","1,013",205,10,"1,297",0,39,130
5,"Census Tract 5, Hudson County, New Jersey","4,311 (r46395)","1,928",977,224,10,"1,062",0,48,62


In [165]:
#clean column names
df2010.columns = df2010.columns.str.replace('\xa0', '', regex=False)
print(df2010.columns.tolist())

['Label (Grouping)', 'Total:', 'Hispanic or Latino', 'White alone', 'Black or African American alone', 'American Indian and Alaska Native alone', 'Asian alone', 'Native Hawaiian and Other Pacific Islander alone', 'Some Other Race alone', 'Two or More Races:']


In [166]:
df2010.head()

,Label (Grouping),Total:,Hispanic or Latino,White alone,Black or African American alone,American Indian and Alaska Native alone,Asian alone,Native Hawaiian and Other Pacific Islander alone,Some Other Race alone,Two or More Races:
1,"Census Tract 1, Hudson County, New Jersey","6,025","2,458","1,437",253,15,"1,693",9,32,128
2,"Census Tract 2, Hudson County, New Jersey","5,409","3,362",987,310,12,630,0,31,77
3,"Census Tract 3, Hudson County, New Jersey","4,220 (r46394)","2,346","1,267",177,10,312,0,35,73
4,"Census Tract 4, Hudson County, New Jersey","3,991","1,297","1,013",205,10,"1,297",0,39,130
5,"Census Tract 5, Hudson County, New Jersey","4,311 (r46395)","1,928",977,224,10,"1,062",0,48,62


In [167]:
#converting string values in columns to floats
df2010['total_pop'] = df2010['Total:'].str.replace(r'\s*\(.*?\)', '', regex=True)
df2010['total_pop'] = df2010['total_pop'].str.replace(',', '', regex=False).astype(float)
df2010['total_white'] = df2010['White alone'].str.replace(',', '', regex=False).astype(float)
df2010['total_black'] = df2010['Black or African American alone'].str.replace(',', '', regex=False).astype(float)
df2010['total_asian'] = df2010['Asian alone'].str.replace(',', '', regex=False).astype(float)
df2010['total_other'] = df2010['Some Other Race alone'].str.replace(',', '', regex=False).astype(float)
df2010['total_hispanic'] = df2010['Hispanic or Latino'].str.replace(',', '', regex=False).astype(float)

In [168]:
#calculating proportions using function
df2010 = race_percentages(df2010, 'total_white', 'total_black', 'total_asian', 'total_other', 'total_hispanic', 'total_pop')

In [160]:
df2010.head()

,Label (Grouping),Total:,Hispanic or Latino,White alone,Black or African American alone,American Indian and Alaska Native alone,Asian alone,Native Hawaiian and Other Pacific Islander alone,Some Other Race alone,Two or More Races:,...,total_white,total_black,total_asian,total_other,total_hispanic,white_pct,black_pct,asian_pct,other_pct,hispanic_pct
1,"Census Tract 1, Hudson County, New Jersey","6,025","2,458","1,437",253,15,"1,693",9,32,128,...,1437.0,253.0,1693.0,32.0,2458.0,23.850622,4.199170,28.099585,0.531120,40.796680
2,"Census Tract 2, Hudson County, New Jersey","5,409","3,362",987,310,12,630,0,31,77,...,987.0,310.0,630.0,31.0,3362.0,18.247366,5.731189,11.647255,0.573119,62.155666
3,"Census Tract 3, Hudson County, New Jersey","4,220 (r46394)","2,346","1,267",177,10,312,0,35,73,...,1267.0,177.0,312.0,35.0,2346.0,30.023697,4.194313,7.393365,0.829384,55.592417
4,"Census Tract 4, Hudson County, New Jersey","3,991","1,297","1,013",205,10,"1,297",0,39,130,...,1013.0,205.0,1297.0,39.0,1297.0,25.382110,5.136557,32.498121,0.977199,32.498121
5,"Census Tract 5, Hudson County, New Jersey","4,311 (r46395)","1,928",977,224,10,"1,062",0,48,62,...,977.0,224.0,1062.0,48.0,1928.0,22.662955,5.196010,24.634656,1.113431,44.722802


In [169]:
#coding near_hblr boolean variable for 2010 dataset
keywords = [' 73,', ' 75,', ' 74,', ' 76,', ' 77,', ' 193,', ' 194,', ' 115,', ' 114,', ' 113,', ' 112,', ' 110,', ' 109,', ' 107,', ' 104,', ' 103,', ' 102,', ' 63,', ' 62,', ' 60,', ' 58.01,', ' 55,', ' 53,', ' 68,', ' 44,', ' 45,', ' 46,', ' 47,', ' 48,', ' 49,', ' 42,', ' 190,', ' 191,', ' 192,', ' 3,', ' 8,', ' 189,', ' 185,', ' 179,', ' 146,', ' 160,', ' 162,', ' 158.02,', ' 161,']
df2010.loc[:, 'near_hblr'] = df2010['Label (Grouping)'].str.contains('|'.join(keywords), na=False)

In [170]:
#coding near_path boolean variable for both 2010 and 2020 datasets based on which tracts are near path
keywords = [' 12.02,', ' 9.02,', ' 71,', ' 19,', ' 20', ' 70,', ' 64,', ' 35,', ' 75,', ' 74,', ' 76,', ' 77,', ' 193,', ' 194,', ]
df2010.loc[:, 'near_path'] = df2010['Label (Grouping)'].str.contains('|'.join(keywords), na=False)
keywords = [' 12.02;', ' 9.02;', ' 71;', ' 19;', ' 20.02;', ' 20.01;', ' 70.02;', ' 70.01;', ' 64;', ' 35;', ' 75;', ' 74;', ' 76.01;', ' 76.02;', ' 77.01;', ' 77.02;', ' 77.03;', ' 193;', ' 194;', ]
df2020.loc[:, 'near_path'] = df2020['census_tract'].str.contains('|'.join(keywords), na=False)

In [171]:
#observing 2010 data, looking only at areas near hblr
df2010[~df2010['near_path']].groupby("near_hblr")[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

,near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,31.056908,7.895428,12.544645,46.067599,0.693537
1,True,32.120228,24.248298,7.348456,33.841164,0.645659


In [28]:
#observing 2020 data, looking only at areas near hblr
df2020[~df2020['near_path']].groupby("near_hblr")[["white_pct", "black_pct", "asian_pct", "hispanic_pct", "other_pct"]].mean().reset_index()

,near_hblr,white_pct,black_pct,asian_pct,hispanic_pct,other_pct
0,False,27.341824,7.336390,15.554732,45.416284,1.400840
1,True,30.875789,20.135449,9.640662,35.208084,1.096248


In [ ]:
#Importing 2020 ACS household income data
df2020income = pd.read_csv("../data/ACSST5Y2020.S1903-Data.csv")

In [182]:
df2020income.head()

,GEO_ID,NAME,S1903_C01_001E,S1903_C03_001E
0,Geography,Geographic Area Name,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...,Estimate!!Median income (dollars)!!HOUSEHOLD I...
1,1400000US34017000101,"Census Tract 1.01, Hudson County, New Jersey",848,91300
2,1400000US34017000102,"Census Tract 1.02, Hudson County, New Jersey",1135,57841
3,1400000US34017000200,"Census Tract 2, Hudson County, New Jersey",1933,39665
4,1400000US34017000300,"Census Tract 3, Hudson County, New Jersey",1477,61069


In [187]:
#initial data cleaning using same function
df2020income = transpose_reset(df2020income, False)
df2020income.head()

,0,Geography,Geographic Area Name,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households,Estimate!!Median income (dollars)!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households
1,1,1400000US34017000101,"Census Tract 1.01, Hudson County, New Jersey",848,91300
2,2,1400000US34017000102,"Census Tract 1.02, Hudson County, New Jersey",1135,57841
3,3,1400000US34017000200,"Census Tract 2, Hudson County, New Jersey",1933,39665
4,4,1400000US34017000300,"Census Tract 3, Hudson County, New Jersey",1477,61069
5,5,1400000US34017000400,"Census Tract 4, Hudson County, New Jersey",1246,80324


In [188]:
#printing columns to confirm no cleaning necessary
print(df2020income.columns.tolist())

[np.int64(0), 'Geography', 'Geographic Area Name', 'Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households', 'Estimate!!Median income (dollars)!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households']


In [189]:
#renaming census tract column for consistency
df2020income.rename(columns={'Geographic Area Name': 'census_tract'}, inplace=True)

In [190]:
#converting household incomes to floats and creating new column to store the values
df2020income['median_income_2020'] = pd.to_numeric(
    df2020income['Estimate!!Median income (dollars)!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households'].str.replace(',', ''), 
    errors='coerce'
)

In [191]:
#renaming household number column
df2020income.rename(columns={'Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households': 'household_num'}, inplace=True)

In [192]:
#converting numerical data for households from strings to floats
df2020income['household_num'] = pd.to_numeric(
    df2020income['household_num'].astype(str).str.replace(',', ''), 
    errors='coerce'
)

In [193]:
df2020income.head()

,0,Geography,census_tract,household_num,Estimate!!Median income (dollars)!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households,median_income_2020
1,1,1400000US34017000101,"Census Tract 1.01, Hudson County, New Jersey",848,91300,91300.0
2,2,1400000US34017000102,"Census Tract 1.02, Hudson County, New Jersey",1135,57841,57841.0
3,3,1400000US34017000200,"Census Tract 2, Hudson County, New Jersey",1933,39665,39665.0
4,4,1400000US34017000300,"Census Tract 3, Hudson County, New Jersey",1477,61069,61069.0
5,5,1400000US34017000400,"Census Tract 4, Hudson County, New Jersey",1246,80324,80324.0


In [ ]:
#pullin 2010 income data
df2010income = pd.read_csv("../data/ACSST5Y2010.S1903-2026-06-01T055927.csv")

In [195]:
df2010income.head()

,Label (Grouping),"Census Tract 1, Hudson County, New Jersey!!Total!!Estimate","Census Tract 1, Hudson County, New Jersey!!Total!!Margin of Error","Census Tract 1, Hudson County, New Jersey!!Median income (dollars)!!Estimate","Census Tract 1, Hudson County, New Jersey!!Median income (dollars)!!Margin of Error","Census Tract 2, Hudson County, New Jersey!!Total!!Estimate","Census Tract 2, Hudson County, New Jersey!!Total!!Margin of Error","Census Tract 2, Hudson County, New Jersey!!Median income (dollars)!!Estimate","Census Tract 2, Hudson County, New Jersey!!Median income (dollars)!!Margin of Error","Census Tract 3, Hudson County, New Jersey!!Total!!Estimate",...,"Census Tract 201, Hudson County, New Jersey!!Median income (dollars)!!Estimate","Census Tract 201, Hudson County, New Jersey!!Median income (dollars)!!Margin of Error","Census Tract 324, Hudson County, New Jersey!!Total!!Estimate","Census Tract 324, Hudson County, New Jersey!!Total!!Margin of Error","Census Tract 324, Hudson County, New Jersey!!Median income (dollars)!!Estimate","Census Tract 324, Hudson County, New Jersey!!Median income (dollars)!!Margin of Error","Census Tract 9801, Hudson County, New Jersey!!Total!!Estimate","Census Tract 9801, Hudson County, New Jersey!!Total!!Margin of Error","Census Tract 9801, Hudson County, New Jersey!!Median income (dollars)!!Estimate","Census Tract 9801, Hudson County, New Jersey!!Median income (dollars)!!Margin of Error"
0,HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATIN...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Households,"2,102",±153,"56,389","±11,847","1,900",±162,"38,438","±8,605","1,568",...,"111,000","±43,375","1,996",±140,"37,234","±8,456",0.0,±123,-,**


In [196]:
#initial cleaning using function
df2010income = transpose_reset(df2010income)
df2010income.head()

,Label (Grouping),HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER,Households
1,"Census Tract 1, Hudson County, New Jersey!!Tot...",NaN,"2,102"
2,"Census Tract 1, Hudson County, New Jersey!!Tot...",NaN,±153
3,"Census Tract 1, Hudson County, New Jersey!!Med...",NaN,"56,389"
4,"Census Tract 1, Hudson County, New Jersey!!Med...",NaN,"±11,847"
5,"Census Tract 2, Hudson County, New Jersey!!Tot...",NaN,"1,900"


In [197]:
#renaming census tract column for consistency
df2010income.rename(columns={'Label (Grouping)': 'census_tract'}, inplace=True)

In [198]:
#Filtering to eliminate median error rows
df10inc = df2010income[df2010income['census_tract'].str.contains("Median income (dollars)!!Estimate", regex=False)]

In [199]:
#dropping unnecessary NaN column
df10inc.drop(columns=['HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER'], inplace=True)

#converting household income into floats
df10inc['median_income_2010'] = pd.to_numeric(
    df10inc['Households'].str.replace(',', ''), 
    errors='coerce'
)

C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\1948144557.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df10inc.drop(columns=['HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER'], inplace=True)
C:\Users\ryanm\AppData\Local\Temp\ipykernel_37272\1948144557.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df10inc['median_income_2010'] = pd.to_numeric(


In [200]:
df10inc.head()

,census_tract,Households,median_income_2010
3,"Census Tract 1, Hudson County, New Jersey!!Med...","56,389",56389.0
7,"Census Tract 2, Hudson County, New Jersey!!Med...","38,438",38438.0
11,"Census Tract 3, Hudson County, New Jersey!!Med...","46,033",46033.0
15,"Census Tract 4, Hudson County, New Jersey!!Med...","70,586",70586.0
19,"Census Tract 5, Hudson County, New Jersey!!Med...","52,147",52147.0


In [ ]:
#pulling additional ACS datasets
df10rent = pd.read_csv("../data//ACSDT5YSPT2010.B25064-2026-06-02T234451.csv")
df20rent = pd.read_csv("../data//ACSDT5Y2020.B25064-2026-06-02T234432.csv")
df10hs = pd.read_csv("../data//ACSDT5Y2010.B25077-2026-06-02T234558.csv")
df20hs = pd.read_csv("../data//ACSDT5Y2020.B25077-2026-06-02T234543.csv")

In [202]:
#creating list in order to create a loop with the ACS cleaning function
dfs = {'df10rent': df10rent, 'df20rent': df20rent, 'df10hs': df10hs, 'df20hs': df20hs}

#looping through list of datasets and applying cleaning function
for name, df in dfs.items():
    dfs[name] = transform_ACS_df(df)

#retrieving datasets from list
df10rent, df20rent, df10hs, df20hs = dfs.values()

#converting values for each dataset into floats
df10hs['median_house_2010'] = pd.to_numeric(
    df10hs['Median value (dollars)'].astype(str).str.replace(',', ''),
    errors='coerce'
)
df20hs['median_house_2020'] = pd.to_numeric(
    df20hs['Median value (dollars)'].astype(str).str.replace(',', ''),
    errors='coerce'
)
df10rent['median_rent_2010'] = pd.to_numeric(
    df10rent['Median gross rent'].astype(str).str.replace(',', '').str.replace('+', ''),
    errors='coerce'
)
df20rent['median_rent_2020'] = pd.to_numeric(
    df20rent['Median gross rent'].astype(str).str.replace(',', '').str.replace('+', ''),
    errors='coerce'
)

In [203]:
df10hs.head()
df20rent.head()

,census_tract,Median value (dollars),tract_num,median_house_2010
1,"Census Tract 1, Hudson County, New Jersey!!Est...","342,900",1.0,342900.0
3,"Census Tract 2, Hudson County, New Jersey!!Est...","275,500",2.0,275500.0
5,"Census Tract 3, Hudson County, New Jersey!!Est...","409,300",3.0,409300.0
7,"Census Tract 4, Hudson County, New Jersey!!Est...","366,000",4.0,366000.0
9,"Census Tract 5, Hudson County, New Jersey!!Est...","413,200",5.0,413200.0


,census_tract,Median gross rent,tract_num,median_rent_2020
1,"Census Tract 1.01, Hudson County, New Jersey!!...","1,659",1.01,1659.0
3,"Census Tract 1.02, Hudson County, New Jersey!!...","1,461",1.02,1461.0
5,"Census Tract 2, Hudson County, New Jersey!!Est...","1,314",2.00,1314.0
7,"Census Tract 3, Hudson County, New Jersey!!Est...","1,288",3.00,1288.0
9,"Census Tract 4, Hudson County, New Jersey!!Est...","1,492",4.00,1492.0


In [204]:
#Exporting cleaned datasets
df2020.to_csv('../data/2020cleaned1.csv', index=False)
df2010.to_csv('../data/2010cleaned1.csv', index=False)
df20hs.to_csv('../data/2020housing.csv', index=False)
df10hs.to_csv('../data/2010housing.csv', index=False)
df20rent.to_csv('../data/2020rent.csv', index=False)
df10rent.to_csv('../data/2010rent.csv', index=False)
df2020income.to_csv('../data/2020income.csv', index=False)
df10inc.to_csv('../data/2010income.csv', index=False)